In [ ]:
import asyncio
import os
import pandas as pd
from tabulate import tabulate
from fastmcp import Client
from fastmcp.client.transports import StreamableHttpTransport
from langchain_core.tools import StructuredTool as _StructuredTool
from langchain.chat_models import init_chat_model
from langgraph.prebuilt import create_react_agent
from typing import TypedDict
import json
import re
from difflib import get_close_matches
from datetime import datetime
import warnings

# Optional: silence Pydantic v2 deprecation warning emitted by LangChain
try:
    from pydantic.warnings import PydanticDeprecatedSince20
    warnings.filterwarnings("ignore", category=PydanticDeprecatedSince20)
except Exception:
    pass

# -----------------------------
# Config (Jupyter-friendly)
# -----------------------------
EXPORT_DIR = os.getenv("EXPORT_DIR", "exports")
# Optionally specify a base URL, e.g., "http://localhost:8888"
JUPYTER_BASE_URL = os.getenv("JUPYTER_BASE_URL", "").rstrip("/")

# -----------------------------
# StructuredTool shim (avoid deprecated __fields__)
# -----------------------------
class StructuredTool(_StructuredTool):
    @classmethod
    def from_function(cls, func, **kwargs):
        tool = super().from_function(func, **kwargs)
        _ = getattr(tool.args_schema, "model_fields", {})
        return tool

# -----------------------------
# Define state schema (optional)
# -----------------------------
class AgentState(TypedDict):
    user_input: str
    agent_response: str

# -----------------------------
# Initialize LLM
# -----------------------------
model = init_chat_model(
    "ollama:qwen2.5:3b",
    model_kwargs={"temperature": 0, "num_predict": 512},
    base_url="http://ollama:11434"
)

# -----------------------------
# Notebook & path helpers
# -----------------------------
def _jupyter_files_url(abs_path: str) -> str | None:
    """
    Build a '/files/...' URL if file is under the current working directory.
    """
    try:
        root = os.getcwd()
        rel = os.path.relpath(abs_path, root)
        if rel.startswith(".."):
            return None
        return "/files/" + rel.replace(os.sep, "/")
    except Exception:
        return None

def ask_export_dir(default_dir: str) -> str:
    """
    Ask the user where to save:
      - 'default' or empty -> use default_dir
      - any other input    -> use that as the directory
    Creates the directory if it doesn't exist.
    """
    while True:
        inp = input(
            f"📁 Save to which directory? Type 'default' to use '{default_dir}', or enter a path: "
        ).strip()
        if not inp or inp.lower() == "default":
            path = default_dir
        else:
            path = inp
        try:
            os.makedirs(path, exist_ok=True)
            return path
        except Exception as e:
            print(f"❌ Couldn’t use that path: {e}. Please try again.")

# -----------------------------
# Excel/CSV save helpers
# -----------------------------
def _get_excel_engine() -> str | None:
    try:
        import openpyxl  # noqa: F401
        return "openpyxl"
    except Exception:
        pass
    try:
        import xlsxwriter  # noqa: F401
        return "xlsxwriter"
    except Exception:
        pass
    return None

def _save_full_to_excel(columns: list, rows: list, suggested_name: str | None, output_dir: str) -> str:
    """
    Save full results to .xlsx if engine available; else CSV.
    Returns absolute path.
    """
    os.makedirs(output_dir, exist_ok=True)
    df = pd.DataFrame(rows, columns=columns)
    stamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    base = suggested_name or "export"
    engine = _get_excel_engine()

    if engine:
        fname = f"export_{base}_{stamp}.xlsx"
        full_path = os.path.join(output_dir, fname)
        with pd.ExcelWriter(full_path, engine=engine) as writer:
            df.to_excel(writer, index=False)
        return os.path.abspath(full_path)
    else:
        fname = f"export_{base}_{stamp}.csv"
        full_path = os.path.join(output_dir, fname)
        df.to_csv(full_path, index=False)
        print("⚠️ Neither 'openpyxl' nor 'XlsxWriter' is installed. Saved CSV instead.")
        print("   To get .xlsx, install one of:\n   - pip install openpyxl\n   - pip install XlsxWriter")
        return os.path.abspath(full_path)

# -----------------------------
# MCP tool wrappers
# -----------------------------
def make_sql_tool_wrapper(tool_name, client):
    def wrapper(sql: str = "") -> str:
        try:
            sql = (sql or "").rstrip(";").strip()
            loop = asyncio.get_event_loop()
            TIMEOUT = int(os.getenv("MCP_CLIENT_TIMEOUT", 60))
            coro = client.call_tool(tool_name, {"sql": sql})
            result = loop.run_until_complete(asyncio.wait_for(coro, timeout=TIMEOUT))
            if hasattr(result, 'data') and result.data:
                return json.dumps(result.data)
            elif hasattr(result, "structuredContent") and result.structuredContent:
                return json.dumps(result.structuredContent)
            elif hasattr(result, "content") and result.content:
                return result.content[0].text if isinstance(result.content, list) else str(result.content)
            else:
                return str(result)
        except asyncio.TimeoutError:
            return f"❌ Timeout calling {tool_name}"
        except Exception as e:
            return f"❌ Error calling {tool_name}: {str(e)}"
    return wrapper

def make_noarg_tool_wrapper(tool_name, client):
    def wrapper() -> str:
        try:
            loop = asyncio.get_event_loop()
            TIMEOUT = int(os.getenv("MCP_CLIENT_TIMEOUT", 60))
            coro = client.call_tool(tool_name, {})
            result = loop.run_until_complete(asyncio.wait_for(coro, timeout=TIMEOUT))
            if hasattr(result, 'data') and result.data:
                return json.dumps(result.data)
            elif hasattr(result, "structuredContent") and result.structuredContent:
                return json.dumps(result.structuredContent)
            elif hasattr(result, "content") and result.content:
                return result.content[0].text if isinstance(result.content, list) else str(result.content)
            else:
                return str(result)
        except asyncio.TimeoutError:
            return f"❌ Timeout calling {tool_name}"
        except Exception as e:
            return f"❌ Error calling {tool_name}: {str(e)}"
    return wrapper

# -----------------------------
# Rendering
# -----------------------------
def display_tables_list(data: dict):
    tables = data.get("tables", [])
    total = data.get("total_tables", len(tables))
    if not tables:
        print("ℹ️ No tables found.")
        return
    print("\n📚 Available tables:")
    for i, t in enumerate(tables, start=1):
        print(f"  {i:>2}. {t}")
    print(f"\nTotal: {total} table(s)")

def display_table_from_json(data):
    preview_rows = data.get("preview_rows", [])
    preview_cols = data.get("preview_columns", [])
    total_rows = data.get("total_rows", len(preview_rows))
    total_cols = data.get("total_columns", len(preview_cols))
    if preview_rows and preview_cols:
        df = pd.DataFrame(preview_rows, columns=preview_cols)
        df.index = df.index + 1
        if total_rows > len(preview_rows):
            df.loc["..."] = ["..."] * len(df.columns)
        if total_cols > len(preview_cols):
            df["..."] = "..."
        print("\n📊 Query Result (preview):")
        print(tabulate(df, headers="keys", tablefmt="pretty"))
        print(f"\nShowing preview: {len(preview_rows)} rows × {len(preview_cols)} columns "
              f"(full result: {total_rows} rows × {total_cols} columns)")
        dbg = data.get("_debug_info", {})
        if dbg:
            print(f"🔍 Debug: Query returned {dbg.get('query_returned_rows', '?')} rows, "
                  f"True total: {dbg.get('true_total_rows', '?')} rows")
    else:
        print("ℹ️ Query executed but returned no rows.")

# -----------------------------
# Parse tool messages
# -----------------------------
def _extract_json_from_text(text: str):
    if not text:
        return None
    try:
        return json.loads(text)
    except Exception:
        pass
    json_pattern = r'\{[^{}]*"success"[^{}]*\}'
    for match in re.findall(json_pattern, text):
        try:
            return json.loads(match)
        except Exception:
            continue
    brace_count = 0
    start_idx = -1
    for i, ch in enumerate(text):
        if ch == '{':
            if brace_count == 0:
                start_idx = i
            brace_count += 1
        elif ch == '}':
            brace_count -= 1
            if brace_count == 0 and start_idx != -1:
                try:
                    parsed = json.loads(text[start_idx:i+1])
                    if isinstance(parsed, dict) and 'success' in parsed:
                        return parsed
                except Exception:
                    pass
    return None

def _collect_tool_payloads(result):
    messages = getattr(result, "messages", []) if hasattr(result, "messages") else result.get("messages", [])
    list_tables_payload, query_db_payload, query_db_sql = None, None, None
    for msg in messages:
        msg_type = getattr(msg, 'type', None) or str(type(msg).__name__).lower()
        if 'tool' in msg_type:
            name = getattr(msg, 'name', None)
            content = getattr(msg, 'content', None)
            args = getattr(msg, 'args', {}) or getattr(msg, 'tool_input', {})
            if name == 'query_db' and args and 'sql' in args:
                query_db_sql = args['sql']
            if isinstance(content, str):
                payload = _extract_json_from_text(content)
            elif isinstance(content, list) and content:
                if isinstance(content[0], dict) and "text" in content[0]:
                    payload = _extract_json_from_text(content[0]["text"])
                else:
                    payload = _extract_json_from_text(str(content[0]))
            else:
                payload = None
            if payload and payload.get("success"):
                if name == 'list_tables':
                    list_tables_payload = payload
                elif name == 'query_db':
                    query_db_payload = payload
    return list_tables_payload, query_db_payload, query_db_sql

def _pick_closest_table(user_input: str, tables: list[str]) -> str | None:
    tokens = re.findall(r"[A-Za-z_][A-Za-z0-9_]*", user_input.lower())
    if not tokens:
        return None
    candidate = tokens[-1]
    bare = [t.split(".", 1)[1] if "." in t else t for t in tables]
    best = get_close_matches(candidate, bare, n=1, cutoff=0.6)
    if best:
        return best[0]
    best = get_close_matches(candidate, tables, n=1, cutoff=0.6)
    if best:
        return best[0].split(".", 1)[1] if "." in best[0] else best[0]
    return None

# -----------------------------
# Agent instructions
# -----------------------------
system_prompt = """You are a SQL assistant that helps users query a PostgreSQL database.

WORKFLOW (you must follow this on every user query):
1) Call `list_tables` first (no arguments).
2) If the user's table name isn't an exact match, choose the closest table from the list (e.g., "department" → "departments").
3) Then call `query_db(sql: str)` using the exact table name. Prefer: SELECT * FROM <table> LIMIT 5;
4) If there is truly no close match, state that no similar table exists and show the available tables.

RULES:
- Only SELECT queries are allowed.
- Always show the SQL you executed.
- Never guess column or table names beyond what `list_tables` returns.
"""

# -----------------------------
# Main
# -----------------------------
async def main():
    transport = StreamableHttpTransport("http://mcp_server:8000/mcp")
    async with Client(transport=transport) as client:

        tools_list = await client.list_tools()
        tools = []
        for tool in tools_list:
            tool_name = getattr(tool, "name", None)
            description = getattr(tool, "description", f"MCP tool {tool_name}")
            if tool_name == "list_tables":
                tool_func = make_noarg_tool_wrapper(tool_name, client)
            else:
                tool_func = make_sql_tool_wrapper(tool_name, client)
            structured_tool = StructuredTool.from_function(
                func=tool_func,
                name=tool_name,
                description=description
            )
            tools.append(structured_tool)

        print("\n🔧 Available MCP Tools:")
        for t in tools:
            print(f"- {t.name}: {t.description}")

        agent_llm = create_react_agent(model=model, tools=tools)

        print("\n🤖 Agent ready! Ask SQL questions.")
        print("Type 'exit', 'quit', or 'q' to stop.")

        while True:
            try:
                user_input = input("\nEnter your SQL query: ").strip()
                if user_input.lower() in ("exit", "quit", "q"):
                    print("👋 Goodbye!")
                    break
                if not user_input:
                    continue

                print("🤔 Thinking...")
                result = await agent_llm.ainvoke({"messages": [("system", system_prompt), ("user", user_input)]})

                list_tables_payload, query_db_payload, query_db_sql = _collect_tool_payloads(result)

                if list_tables_payload:
                    display_tables_list(list_tables_payload)

                auto_sql_used = None
                if query_db_payload:
                    if query_db_sql:
                        print(f"\n🔍 SQL executed (by agent): {query_db_sql}")
                    display_table_from_json(query_db_payload)
                else:
                    if list_tables_payload and list_tables_payload.get("tables"):
                        tables = list_tables_payload["tables"]
                        best = _pick_closest_table(user_input, tables)
                        if best:
                            auto_sql_used = f"SELECT * FROM {best} LIMIT 5"
                            print(f"\n🤖 No follow-up from agent. Auto-querying closest match: {best}")
                            print(f"🔍 SQL executed (auto): {auto_sql_used}")
                            auto = await client.call_tool("query_db", {"sql": auto_sql_used})
                            payload = auto.data if hasattr(auto, 'data') else (
                                auto.structuredContent if hasattr(auto, 'structuredContent') else (
                                    auto.content[0].text if hasattr(auto, "content") and auto.content else None
                                )
                            )
                            if isinstance(payload, str):
                                try:
                                    payload = json.loads(payload)
                                except Exception:
                                    payload = None
                            if isinstance(payload, dict) and payload.get("success"):
                                query_db_payload = payload
                                display_table_from_json(query_db_payload)
                            else:
                                print("ℹ️ Auto query did not return a valid payload.")

                # Ask for Excel export ONLY after a successful preview
                if query_db_payload and query_db_payload.get("success"):
                    dbg = query_db_payload.get("_debug_info", {}) or {}
                    base_query = dbg.get("base_query") or query_db_sql or auto_sql_used or ""
                    ans = input("\n💾 Export the FULL result to Excel (it won't be printed here)? [y/N]: ").strip().lower()
                    if ans in ("y", "yes"):
                        print("📥 Exporting full result, please wait...")
                        full = await client.call_tool("query_db_full", {"sql": base_query})
                        full_payload = full.data if hasattr(full, 'data') else (
                            full.structuredContent if hasattr(full, 'structuredContent') else (
                                full.content[0].text if hasattr(full, "content") and full.content else None
                            )
                        )
                        if isinstance(full_payload, str):
                            try:
                                full_payload = json.loads(full_payload)
                            except Exception:
                                full_payload = None
                        if isinstance(full_payload, dict) and full_payload.get("success"):
                            cols = full_payload.get("columns", [])
                            rows = full_payload.get("rows", [])
                            m = re.search(r'from\s+("?([A-Za-z0-9_]+)"?\.)?"?([A-Za-z0-9_]+)"?', base_query, flags=re.IGNORECASE)
                            suggested = m.group(3) if m else None

                            # Ask where to save: 'default' or a custom path
                            export_dir = ask_export_dir(EXPORT_DIR)
                            outpath = _save_full_to_excel(cols, rows, suggested_name=suggested, output_dir=export_dir)

                            print(f"✅ Saved: {outpath}")

                            # Show a Jupyter /files/ URL if the file is under notebook root
                            rel_url = _jupyter_files_url(outpath)
                            if rel_url:
                                full_url = f"{JUPYTER_BASE_URL}{rel_url}" if JUPYTER_BASE_URL else rel_url
                                print(f"⬇️  Download via Jupyter: {full_url}")
                            else:
                                print("📁 The file is outside the Jupyter root; open it from the file system.")

                            if full_payload.get("truncated"):
                                print("⚠️ Export was truncated to the server cap.")
                        else:
                            print("❌ Failed to export full result.")

                # Optional: final AI message
                messages = getattr(result, "messages", [])
                final_response = None
                for msg in reversed(messages):
                    msg_type = getattr(msg, 'type', None) or str(type(msg).__name__).lower()
                    if 'ai' in msg_type and hasattr(msg, 'content') and msg.content:
                        final_response = msg.content
                        break
                if final_response:
                    print("\n🤖 Agent Response:")
                    print(final_response)

            except KeyboardInterrupt:
                print("\n👋 Goodbye!")
                break
            except Exception as e:
                print(f"\n❌ Unexpected error: {e}")
                import traceback
                print("Debug traceback:")
                traceback.print_exc()

# -----------------------------
# Entry
# -----------------------------
if __name__ == "__main__":
    try:
        asyncio.run(main())
    except RuntimeError as e:
        if "asyncio.run() cannot be called" in str(e):
            import nest_asyncio
            nest_asyncio.apply()
            loop = asyncio.get_event_loop()
            loop.run_until_complete(main())
        else:
            raise



🔧 Available MCP Tools:
- list_tables: List all table names in the connected PostgreSQL database.
- query_db: Return a safe server-side 5x5 preview + counts for a SELECT query.
- query_db_full: Execute the SELECT (without LIMIT) and return ALL rows (capped by MAX_EXPORT_ROWS).
Intended for file export only—do not print these rows on client.

🤖 Agent ready! Ask SQL questions.
Type 'exit', 'quit', or 'q' to stop.



Enter your SQL query:  display me the content of department table


🤔 Thinking...

📊 Query Result (preview):
+-----+---------------+-----------------+------------+------------+-----------+-----+
|     | department_id | department_name | manager_id |  location  |  budget   | ... |
+-----+---------------+-----------------+------------+------------+-----------+-----+
|  1  |       1       |  Department 1   |     38     | Location 7 | 365812.04 | ... |
|  2  |       2       |  Department 2   |     48     | Location 4 | 202327.53 | ... |
|  3  |       3       |  Department 3   |     43     | Location 4 | 853790.84 | ... |
|  4  |       4       |  Department 4   |     21     | Location 4 | 19482.89  | ... |
|  5  |       5       |  Department 5   |     49     | Location 8 | 619989.13 | ... |
| ... |      ...      |       ...       |    ...     |    ...     |    ...    | ... |
+-----+---------------+-----------------+------------+------------+-----------+-----+

Showing preview: 5 rows × 5 columns (full result: 50 rows × 7 columns)
🔍 Debug: Query returned 5 r


💾 Export the FULL result to Excel (it won't be printed here)? [y/N]:  y


📥 Exporting full result, please wait...


📁 Save to which directory? Type 'default' to use 'exports', or enter a path:  


... (3 outputs left out)


💾 Export the FULL result to Excel (it won't be printed here)? [y/N]:  y


📥 Exporting full result, please wait...


📁 Save to which directory? Type 'default' to use 'exports', or enter a path:  testing


✅ Saved: /app/testing/export_employees_2025-09-09_05-58-35.xlsx
⬇️  Download via Jupyter: /files/testing/export_employees_2025-09-09_05-58-35.xlsx
